# 02. 학습률에 따른 검증 손실 비교


## 0. 작업 전 준비사항
이 실습은 작은 데이터를 사용하므로 CPU로 충분합니다. 아래 코드는 모델과 텐서를 CPU에 두며, CUDA를 사용할 수 있다는 사실만으로 GPU에서 실행되지는 않습니다.

GPU 환경을 확인하고 싶다면 다음 항목을 점검합니다.
- [ ] GPU 모델 및 드라이버 버전 확인 (`nvidia-smi`, NVIDIA GPU 환경)
- [ ] PyTorch 버전 확인 (`torch.__version__`)
- [ ] 설치된 PyTorch의 CUDA 빌드 버전 확인 (`torch.version.cuda`, CPU 빌드는 `None`)
- [ ] PyTorch의 CUDA 사용 가능 여부 확인 (`torch.cuda.is_available()`)

GPU로 학습하려면 모델과 입력·정답 텐서를 모두 같은 장치로 이동해야 합니다. 이번 비교에서는 CPU 설정을 그대로 사용합니다.


## 1. 파일 불러오기 및 학습·검증 텐서 준비
AdamW의 학습률 `0.001`과 `0.01`에 따른 학습 결과를 비교하기 위해 다음 과정을 거칩니다.

- `points.csv`를 읽고 `split` 열에 따라 학습용 `train`과 검증용 `valid`를 구분합니다.
- 입력 `X`는 `x1`, `x2` 두 특성을 가진 **2차원 텐서 `(샘플 수, 2)`**이며 `float32`로 만듭니다.
- 정답 `y`는 클래스 번호를 담은 **1차원 텐서 `(샘플 수,)`**이며 `long`으로 만듭니다.
- `test` 데이터는 이번 학습과 검증에 사용하지 않습니다. 검증 데이터로 설정을 고른 뒤 최종 평가에 사용할 데이터입니다.

아래 경로 코드는 저장소 루트, `01_기본필수`, `docs` 중 어느 위치에서 노트북을 실행해도 데이터 파일을 찾도록 구성했습니다. Jupyter에서는 일반적으로 `__file__`이 정의되지 않으므로 현재 작업 폴더를 기준으로 찾습니다.


In [ ]:
from pathlib import Path
import pandas as pd
import torch
from torch import nn

torch.set_num_threads(1)
candidates = [
    Path.cwd() / "points.csv",
    Path.cwd().parent / "points.csv",
    Path.cwd() / "20261030" / "01_기본필수" / "points.csv",
]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("저장소 루트, 01_기본필수 또는 docs 폴더에서 실행하세요.")
df = pd.read_csv(data_path)
train = df[df["split"] == "train"]
valid = df[df["split"] == "valid"]
X_train = torch.tensor(train[["x1", "x2"]].values, dtype=torch.float32)
y_train = torch.tensor(train["label"].values, dtype=torch.long)
X_valid = torch.tensor(valid[["x1", "x2"]].values, dtype=torch.float32)
y_valid = torch.tensor(valid["label"].values, dtype=torch.long)


## 2. 학습률(Learning Rate)의 역할
학습률 `lr`은 Optimizer가 파라미터를 갱신할 때 사용하는 기본 보폭입니다. 모델이 학습하는 가중치와 달리 실험자가 정하는 **하이퍼파라미터**입니다.

기본 경사하강법에서는 다음과 같이 작용합니다. $\theta$는 파라미터, $\eta$는 학습률입니다.

$$\theta_{t+1} = \theta_t - \eta \nabla_{\theta} L(\theta_t)$$

AdamW는 기울기의 1차·2차 모멘트 추정치로 갱신 방향과 크기를 조절하고, 여기에 기본 학습률을 적용합니다. 따라서 `lr=0.01`이라고 해서 각 가중치가 매번 정확히 `0.01`씩 바뀌는 것은 아닙니다.

| 학습률 설정 | 관찰할 수 있는 현상 | 해석 시 주의점 |
| --- | --- | --- |
| 너무 작음 | 정해진 횟수 안에 손실이 조금씩만 감소 | 학습 횟수가 부족한 것일 수도 있음 |
| 적절함 | 손실이 비교적 안정적으로 감소 | 검증 성능도 함께 확인해야 함 |
| 너무 큼 | 손실이 진동하거나 커지고 발산할 수 있음 | 큰 학습률이 항상 발산하는 것은 아님 |

이번 후보는 `0.001`과 `0.01`로 10배 차이가 납니다. 어느 값이 더 적절한지는 실제 검증 손실을 비교해 판단합니다. 각 실험 중에는 학습률을 고정하며, 학습 도중 값을 변경하는 Scheduler는 사용하지 않습니다.


## 3. 학습률만 바꾸어 학습하고 검증하기
### 비교 조건
- **변경:** `learning_rates = [0.001, 0.01]`
- **고정:** 데이터, `seed=42`, 모델 구조, AdamW, `weight_decay=0.0`, 20 epochs
- 각 후보마다 seed를 다시 지정하고 모델과 Optimizer를 새로 생성합니다. 같은 초기 가중치에서 시작하며 AdamW의 내부 상태도 초기화됩니다.
- `Linear(2, 8) → ReLU → Linear(8, 2)`의 출력은 logits입니다. `CrossEntropyLoss`에 넣기 전에 Softmax를 따로 적용하지 않습니다.

### epoch마다 실행되는 순서
1. `model.train()`으로 학습 모드를 설정합니다.
2. `optimizer.zero_grad()`로 이전 기울기를 지웁니다.
3. 학습 데이터로 손실을 계산하고 `backward()`로 기울기를 구합니다.
4. `optimizer.step()`으로 파라미터를 갱신합니다.
5. `model.eval()`과 `torch.no_grad()`로 **갱신 후 모델의 검증 손실**을 계산하고 `losses`에 저장합니다.

학습 데이터 전체를 한 번에 사용하므로 **1 epoch = 1 optimizer step**입니다. 첫 기록은 학습 전 초기 손실이 아니라 첫 갱신이 끝난 뒤의 손실입니다. 검증 데이터로 `backward()`나 `step()`을 실행하지 않습니다.

`model.eval()`은 평가 모드를 설정하고, `torch.no_grad()`는 기울기 추적을 끕니다. 두 기능은 별개이며 이 모델에는 Dropout과 BatchNorm이 없지만 일반적인 평가 패턴을 따릅니다.


In [ ]:
# 학습률 후보 두 개를 리스트로 쓰세요: 0.001, 0.01.
# 모델·seed·20 epochs·AdamW·weight_decay=0은 그대로 둡니다.
# >>> B03
# TODO B03: 위 요구사항에 맞게 이 블록만 완성합니다.
learning_rates = [0.001, 0.01]
# <<< B03
loss_fn = nn.CrossEntropyLoss()
for lr in learning_rates:
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    losses = []
    for epoch in range(20):
        model.train()
        optimizer.zero_grad()
        loss_fn(model(X_train), y_train).backward()
        optimizer.step()
        # 갱신이 끝난 모델의 valid loss를 기록합니다. valid로 backward하지 않습니다.
        model.eval()
        with torch.no_grad():
            losses.append(loss_fn(model(X_valid), y_valid).item())
    print("lr:", lr, "처음 5개:", [round(x, 6) for x in losses[:5]])
    print("마지막 valid loss:", round(losses[-1], 6))


## 4. 결과 해석 및 추가 실험
`처음 5개`는 1~5 epoch의 검증 손실이며, `마지막 valid loss`는 20 epoch 후의 검증 손실입니다. `losses[:5]`는 앞의 다섯 항목을, `losses[-1]`은 마지막 항목을 가져옵니다. 출력만 소수점 여섯 자리로 반올림하며 실제 기록값은 반올림하지 않습니다.

같은 검증 데이터와 손실 함수에서는 검증 손실이 작을수록 더 적절한 예측을 했다고 볼 수 있습니다. 교차 엔트로피 손실은 오답 비율이나 백분율이 아니므로 **손실률** 대신 **손실값**이라고 표현합니다. 손실이 줄어도 예측 클래스가 바뀌지 않으면 정확도는 같을 수 있습니다.

### 결과에서 확인할 질문
- 첫 5회의 손실은 어느 학습률에서 더 빠르게 줄어드는가?
- 20회 후의 검증 손실도 같은 후보가 더 작은가?
- 중간에 손실이 증가하는 구간이 있는가? 처음과 끝만으로 전체 흐름을 판단하지 않았는가?

20회 후 더 작은 손실을 보인 후보는 **이번 조건과 학습 예산에서** 더 좋은 결과를 낸 것입니다. 이를 모든 데이터와 학습 횟수에 적용되는 최적 학습률로 일반화하지 않습니다. 검증 손실은 매 epoch 반드시 감소해야 하는 값도 아닙니다.

### 추가 실험
- 학습률 후보에 `0.0001`, `0.1`을 추가하고 감소 속도와 안정성을 비교합니다. `0.1`에서 반드시 발산한다고 가정하지 않습니다.
- 학습 횟수만 100회로 늘려 작은 학습률이 더 많은 갱신 후에도 같은 결과를 보이는지 확인합니다.
- 여러 seed에서 반복하되, 같은 seed의 두 후보는 같은 초기 가중치에서 시작하도록 합니다.
- 학습 손실과 검증 손실을 같은 갱신 후 시점에 함께 기록하면 느린 학습과 과적합을 구분하는 데 도움이 됩니다.

검증 결과로 학습률을 선택하고, `test`는 설정을 결정한 뒤 최종 평가에 사용합니다. 학습률만 비교하려면 이전 모델을 이어 학습하거나 한 후보에만 학습 횟수·가중치 감쇠를 다르게 적용하지 않도록 주의합니다.
